# Notebook 1: QUBO và mô hình Ising

Đi kèm [Phần A, Mục 2.1](../../docs/A-translation/02-background.md). Notebook kiểm tra bằng số các công thức chuyển QUBO sang mô hình Ising, phương trình (8)–(11), trên mọi tổ hợp nhị phân.

Ý tưởng: với phép thế $x_j = (1 - s_j)/2$, hàm QUBO $f(x) = \sum_{j,k} Q_{j,k} x_j x_k + c$ trở thành hàm Ising $f(s) = \sum_{j,k} J_{j,k} s_j s_k + \sum_j h_j s_j$ có cùng giá trị tại các điểm tương ứng.

In [1]:
import itertools, sys
import numpy as np
sys.path.insert(0, "../../code")
from qco_utils import qubo_value, qubo_to_ising, ising_value

## 1. Kiểm tra trên một QUBO ngẫu nhiên

In [2]:
rng = np.random.default_rng(0)
n = 5
A = rng.integers(-5, 6, size=(n, n)).astype(float)
Q = (A + A.T) / 2          # QUBO đối xứng, như giả thiết của bài báo
c = 3.0
J, h = qubo_to_ising(Q, c)

max_err = 0.0
for x in itertools.product((0, 1), repeat=n):
    s = [1 - 2 * xi for xi in x]                  # x = (1 - s)/2  <=>  s = 1 - 2x
    max_err = max(max_err, abs(qubo_value(Q, c, x) - ising_value(J, h, s)))
print("sai số lớn nhất trên 2^5 tổ hợp:", max_err)

sai số lớn nhất trên 2^5 tổ hợp: 0.0


## 2. Kiểm tra trên QUBO của Ví dụ 15 (phương trình (39))

Với $P = 7$ và các biến theo thứ tự $(x_1, x_2, x_3, z)$:

$$f(x, z) = x_1 z - 2z - x_1 x_3 + x_2 + x_3 + P\,(x_2 x_3 - 2x_2 z - 2x_3 z + 3z).$$

Ma trận $Q$ đối xứng nhận được bằng cách chia đôi coefficient của mỗi tích hai biến khác nhau vào hai vị trí $(j,k)$ và $(k,j)$.

In [3]:
P = 7
def f39(x1, x2, x3, z):
    return x1*z - 2*z - x1*x3 + x2 + x3 + P*(x2*x3 - 2*x2*z - 2*x3*z + 3*z)

Q = np.zeros((4, 4))
def add_pair(j, k, coef):            # coefficient của x_j x_k (j != k), chia đều cho (j,k) và (k,j)
    Q[j, k] += coef / 2; Q[k, j] += coef / 2
# thứ tự biến: 0:x1, 1:x2, 2:x3, 3:z
add_pair(0, 3, 1)                    # x1 z
Q[3, 3] += -2 + 3 * P                # -2z + 3P z
add_pair(0, 2, -1)                   # -x1 x3
Q[1, 1] += 1                         # x2
Q[2, 2] += 1                         # x3
add_pair(1, 2, P)                    # P x2 x3
add_pair(1, 3, -2 * P)               # -2P x2 z
add_pair(2, 3, -2 * P)               # -2P x3 z

J, h = qubo_to_ising(Q, 0.0)
err_q = max(abs(qubo_value(Q, 0, x) - f39(*x)) for x in itertools.product((0, 1), repeat=4))
err_i = max(abs(ising_value(J, h, [1 - 2 * xi for xi in x]) - f39(*x)) for x in itertools.product((0, 1), repeat=4))
print("Q khớp (39):", err_q, "| Ising khớp (39):", err_i)
print("J =\n", J)
print("h =", h)

Q khớp (39): 0.0 | Ising khớp (39): 0.0
J =
 [[ 5.25   0.    -0.125  0.125]
 [ 0.     0.     0.875 -1.75 ]
 [-0.125  0.875  0.    -1.75 ]
 [ 0.125 -1.75  -1.75   0.   ]]
h = [-0.    1.25  1.5  -2.75]


Cả hai sai số bằng 0: ma trận $Q$ và mô hình Ising thu được từ (8)–(11) cho đúng giá trị của hàm trong phương trình (39) ở cả 16 tổ hợp. Ghi chú: $J_{1,1}$ chứa toàn bộ hằng số của mô hình (nó chỉ dịch hàm mục tiêu, không đổi optimal solution).